# 00B｜Celeb-DF v2 平衡測試集建立

從 Celeb-DF v2 官方測試集建立一組類別平衡的影像測試集，供 04 與 08 使用。

**處理方式**

1. 讀取官方 `List_of_testing_videos.txt`，共 518 部影片（Real 178、Fake 340）。
2. 保留全部 178 部 Real 影片。
3. 以固定 seed 42 從 340 部 Fake 中抽出 178 部，使兩類影片數相同。抽中的清單另存為 `selected_fake_videos_seed42.txt`，確保結果可完整還原。
4. 每部影片在中間 80% 的範圍內（避開前後各 10%）嘗試 12 個等距位置，取到 3 張成功偵測人臉的影格為止。
5. 以 YOLOv11n-face 取面積最大的人臉，向外擴張 20% 後裁切並縮放為 224×224。

使用官方測試清單而非自行切分，是為了讓結果與其他使用 Celeb-DF v2 的研究維持可比性。類別平衡則是為了讓 accuracy 與 F1 不受類別比例影響——原始清單中 Fake 佔近三分之二，未平衡時這些指標會失去意義。

**輸出**：Real 534 張、Fake 534 張，共 1,068 張。356 部影片全數成功處理，無失敗案例。

**已知限制：裁切方式與訓練資料不一致**

本 Notebook 的 `crop_largest_face` 在擴張人臉框後**未補成正方形即縮放為 224×224**，長寬比不同於 1:1 的人臉框會被壓縮變形。

FF++ 的訓練資料（02）與手機測試資料（09A）皆先將人臉框補成正方形再縮放，比例不失真。因此模型在 Celeb-DF v2 上面對的是與訓練時比例不同的人臉。

此外偵測參數也不同：本 Notebook 使用 conf 0.60、margin 0.20，FF++ 使用 conf 0.50、padding 0.30。

這些差異無法排除為 Celeb-DF v2 上表現較低的部分原因，不應將該結果完全歸因於資料集本身的困難度。

In [1]:
# 第一次執行時取消註解
#%pip install ultralytics opencv-python pandas pillow tqdm


In [2]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib import font_manager

def configure_chinese_font():
    font_candidates = [
        Path(r"C:\Windows\Fonts\msjh.ttc")      # 微軟正黑體
    ]

    for font_path in font_candidates:
        if font_path.exists():
            font_manager.fontManager.addfont(str(font_path))

            font_prop = font_manager.FontProperties(
                fname=str(font_path)
            )
            font_name = font_prop.get_name()

            plt.rcParams["font.family"] = "sans-serif"
            plt.rcParams["font.sans-serif"] = [font_name]
            plt.rcParams["axes.unicode_minus"] = False

            print("已套用中文字體：", font_name)
            print("字體路徑：", font_path)

            return font_prop

    raise FileNotFoundError(
        "找不到 Windows 內建中文字體。"
    )

CHINESE_FONT_PROP = configure_chinese_font()

已套用中文字體： Microsoft JhengHei
字體路徑： C:\Windows\Fonts\msjh.ttc


## 1. 路徑與參數

`RANDOM_SEED = 42` 用於 Fake 影片的下採樣。抽樣結果會另存為文字檔，即使重跑也能確認使用的是同一批影片。

`MAX_FRAMES_PER_VIDEO = 3` 但候選位置有 12 個：部分影格可能偵測不到人臉，多備候選讓每部影片都能取滿 3 張，避免不同影片的取樣數不一致。

In [3]:
from pathlib import Path

# Celeb-DF v2 根目錄
CELEB_DF_ROOT = Path(r"D:\資料\專題\Celeb-DF-v2")

# 官方測試清單
TEST_LIST_PATH = CELEB_DF_ROOT / "List_of_testing_videos.txt"

# YOLO Face 權重
YOLO_FACE_WEIGHT = Path(
    r"D:\資料\專題\PY\deepfake\vit(舊版\yolov11n-face.pt"
)

# 使用新的輸出資料夾，避免和之前不平衡版本混在一起
OUTPUT_ROOT = Path("./CelebDF_v2_balanced_test")

# 固定抽樣設定
RANDOM_SEED = 42

# 每部影片最多保留 3 張人臉圖片
MAX_FRAMES_PER_VIDEO = 3

# 每部影片嘗試 12 個均勻位置，直到成功取得 3 張
CANDIDATE_FRAMES_PER_VIDEO = 12

YOLO_CONF = 0.60
FACE_MARGIN = 0.20
OUTPUT_SIZE = 224

# True：每次執行前清空本 Notebook 產生的圖片與紀錄
# 建議正式重跑時保持 True，避免舊資料混入
RESET_OUTPUT = True

print("Celeb-DF 根目錄：", CELEB_DF_ROOT)
print("官方測試清單：", TEST_LIST_PATH)
print("輸出位置：", OUTPUT_ROOT)
print("固定亂數種子：", RANDOM_SEED)


Celeb-DF 根目錄： D:\資料\專題\Celeb-DF-v2
官方測試清單： D:\資料\專題\Celeb-DF-v2\List_of_testing_videos.txt
輸出位置： CelebDF_v2_balanced_test
固定亂數種子： 42


## 2. 匯入套件與檢查路徑

In [4]:
import random
import shutil
from collections import Counter
from typing import List, Tuple

import cv2
import numpy as np
import pandas as pd
import torch

from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"

if not CELEB_DF_ROOT.exists():
    raise FileNotFoundError(f"找不到 Celeb-DF 根目錄：{CELEB_DF_ROOT}")

if not TEST_LIST_PATH.exists():
    raise FileNotFoundError(f"找不到官方測試清單：{TEST_LIST_PATH}")

if not YOLO_FACE_WEIGHT.exists():
    raise FileNotFoundError(f"找不到 YOLO Face 權重：{YOLO_FACE_WEIGHT}")

detector = YOLO(str(YOLO_FACE_WEIGHT))

print("裝置：", DEVICE)


裝置： cuda:0


## 3. 解析官方清單並建立平衡影片清單

官方清單的標籤定義為 `1 = Real`、`0 = Fake`，與本研究其餘部分使用的 `Real=0`、`Fake=1` 相反。此處依官方定義解析，輸出時改以 `real`／`fake` 資料夾區分，後續 Notebook 的標籤只由資料夾決定，不會受此影響。

程式會先確認清單中的 518 部影片全部存在，任一缺漏即中止。

In [5]:
def parse_official_test_list(
    list_path: Path,
    dataset_root: Path,
) -> List[Tuple[int, Path]]:
    records = []

    with open(list_path, "r", encoding="utf-8-sig") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line:
                continue

            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                raise ValueError(
                    f"第 {line_number} 行格式錯誤：{line}"
                )

            label_text, relative_path = parts
            label = int(label_text)

            if label not in (0, 1):
                raise ValueError(
                    f"第 {line_number} 行標籤不是 0/1：{line}"
                )

            video_path = dataset_root / Path(relative_path)
            records.append((label, video_path))

    return records


def build_balanced_video_records(
    records: List[Tuple[int, Path]],
    seed: int,
):
    real_records = [
        record for record in records if record[0] == 1
    ]
    fake_records = [
        record for record in records if record[0] == 0
    ]

    if len(fake_records) < len(real_records):
        raise ValueError(
            "Fake 影片數少於 Real 影片數，無法依 Real 數量下採樣。"
        )

    rng = random.Random(seed)
    selected_fake_records = rng.sample(
        fake_records,
        len(real_records),
    )

    balanced_records = real_records + selected_fake_records
    rng.shuffle(balanced_records)

    return (
        balanced_records,
        real_records,
        selected_fake_records,
    )


ALL_TEST_RECORDS = parse_official_test_list(
    TEST_LIST_PATH,
    CELEB_DF_ROOT,
)

original_counts = Counter(
    label for label, _ in ALL_TEST_RECORDS
)

print("官方測試影片總數：", len(ALL_TEST_RECORDS))
print("官方 Real（1）：", original_counts[1])
print("官方 Fake（0）：", original_counts[0])

missing_videos = [
    str(path)
    for _, path in ALL_TEST_RECORDS
    if not path.exists()
]

print("找不到的影片數：", len(missing_videos))

if missing_videos:
    print("前 10 個找不到的影片：")
    for item in missing_videos[:10]:
        print(item)

    raise FileNotFoundError(
        "官方測試清單中的部分影片不存在，請確認資料集路徑。"
    )


(
    BALANCED_TEST_RECORDS,
    REAL_RECORDS,
    SELECTED_FAKE_RECORDS,
) = build_balanced_video_records(
    ALL_TEST_RECORDS,
    RANDOM_SEED,
)

balanced_counts = Counter(
    label for label, _ in BALANCED_TEST_RECORDS
)

print()
print("平衡後 Real 影片數：", balanced_counts[1])
print("平衡後 Fake 影片數：", balanced_counts[0])
print("平衡後影片總數：", len(BALANCED_TEST_RECORDS))

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# 儲存抽中的 Fake 影片清單
selected_fake_txt = (
    OUTPUT_ROOT
    / f"selected_fake_videos_seed{RANDOM_SEED}.txt"
)

with open(selected_fake_txt, "w", encoding="utf-8") as file:
    for _, video_path in sorted(
        SELECTED_FAKE_RECORDS,
        key=lambda item: str(item[1]),
    ):
        relative_path = video_path.relative_to(CELEB_DF_ROOT)
        file.write(str(relative_path).replace("\\", "/") + "\n")

# 儲存完整平衡影片清單
balanced_video_df = pd.DataFrame([
    {
        "official_label": label,
        "class_name": "real" if label == 1 else "fake",
        "relative_video_path": str(
            path.relative_to(CELEB_DF_ROOT)
        ).replace("\\", "/"),
        "absolute_video_path": str(path),
    }
    for label, path in BALANCED_TEST_RECORDS
])

balanced_video_df.to_csv(
    OUTPUT_ROOT / "balanced_video_list.csv",
    index=False,
    encoding="utf-8-sig",
)

print()
print("已儲存 Fake 抽樣清單：", selected_fake_txt)
print(
    "已儲存平衡影片清單：",
    OUTPUT_ROOT / "balanced_video_list.csv",
)


官方測試影片總數： 518
官方 Real（1）： 178
官方 Fake（0）： 340
找不到的影片數： 0

平衡後 Real 影片數： 178
平衡後 Fake 影片數： 178
平衡後影片總數： 356

已儲存 Fake 抽樣清單： CelebDF_v2_balanced_test\selected_fake_videos_seed42.txt
已儲存平衡影片清單： CelebDF_v2_balanced_test\balanced_video_list.csv


## 4. 人臉裁切函式

`candidate_frame_indices` 避開影片前後各 10%。片頭與片尾常有轉場、黑幀或不完整的人臉，中段的畫面較穩定。

In [6]:
def candidate_frame_indices(
    total_frames: int,
    count: int,
):
    if total_frames <= 0:
        return []

    # 避開影片最前面與最後面 10%
    start = int(total_frames * 0.10)
    end = max(start + 1, int(total_frames * 0.90))

    return np.linspace(
        start,
        end - 1,
        count,
        dtype=int,
    ).tolist()


def crop_largest_face(frame_bgr):
    result = detector.predict(
        source=frame_bgr,
        conf=YOLO_CONF,
        verbose=False,
        device=DEVICE,
    )[0]

    if result.boxes is None or len(result.boxes) == 0:
        return None

    boxes = result.boxes.xyxy.detach().cpu().numpy()

    areas = (
        (boxes[:, 2] - boxes[:, 0])
        * (boxes[:, 3] - boxes[:, 1])
    )

    x1, y1, x2, y2 = boxes[int(np.argmax(areas))]

    height, width = frame_bgr.shape[:2]
    box_width = x2 - x1
    box_height = y2 - y1

    x1 = max(0, int(x1 - box_width * FACE_MARGIN))
    y1 = max(0, int(y1 - box_height * FACE_MARGIN))
    x2 = min(width, int(x2 + box_width * FACE_MARGIN))
    y2 = min(height, int(y2 + box_height * FACE_MARGIN))

    if x2 <= x1 or y2 <= y1:
        return None

    crop_bgr = frame_bgr[y1:y2, x1:x2]
    crop_rgb = cv2.cvtColor(
        crop_bgr,
        cv2.COLOR_BGR2RGB,
    )

    return Image.fromarray(crop_rgb).resize(
        (OUTPUT_SIZE, OUTPUT_SIZE),
        Image.Resampling.LANCZOS,
    )


## 5. 依平衡影片清單切圖

同時輸出 `extraction_log.csv`，記錄每張圖片的來源影片與影格位置，可回溯任一張測試影像的出處。

In [7]:
def prepare_output_folders():
    test_root = OUTPUT_ROOT / "test"

    if RESET_OUTPUT and test_root.exists():
        shutil.rmtree(test_root)

    output_real = test_root / "real"
    output_fake = test_root / "fake"

    output_real.mkdir(parents=True, exist_ok=True)
    output_fake.mkdir(parents=True, exist_ok=True)

    return output_real, output_fake


def process_balanced_test_set(records):
    output_real, output_fake = prepare_output_folders()

    saved_counts = {
        "real": 0,
        "fake": 0,
    }

    successful_video_counts = {
        "real": 0,
        "fake": 0,
    }

    failed_video_counts = {
        "real": 0,
        "fake": 0,
    }

    log_rows = []

    for label, video_path in tqdm(
        records,
        desc="處理平衡 Celeb-DF 測試影片",
    ):
        class_name = "real" if label == 1 else "fake"
        output_dir = (
            output_real
            if class_name == "real"
            else output_fake
        )

        capture = cv2.VideoCapture(str(video_path))

        if not capture.isOpened():
            failed_video_counts[class_name] += 1
            log_rows.append({
                "official_label": int(label),
                "class_name": class_name,
                "source_video": str(video_path),
                "relative_video_path": str(
                    video_path.relative_to(CELEB_DF_ROOT)
                ).replace("\\", "/"),
                "frame_index": None,
                "saved_image_path": None,
                "status": "video_open_failed",
            })
            continue

        total_frames = int(
            capture.get(cv2.CAP_PROP_FRAME_COUNT)
        )

        frame_indices = candidate_frame_indices(
            total_frames,
            CANDIDATE_FRAMES_PER_VIDEO,
        )

        saved_from_video = 0

        for frame_index in frame_indices:
            if saved_from_video >= MAX_FRAMES_PER_VIDEO:
                break

            capture.set(
                cv2.CAP_PROP_POS_FRAMES,
                frame_index,
            )

            success, frame = capture.read()

            if not success or frame is None:
                continue

            face = crop_largest_face(frame)

            if face is None:
                continue

            filename = (
                f"{class_name}_"
                f"{video_path.stem}_"
                f"f{frame_index:06d}.png"
            )

            save_path = output_dir / filename
            face.save(save_path)

            log_rows.append({
                "official_label": int(label),
                "class_name": class_name,
                "source_video": str(video_path),
                "relative_video_path": str(
                    video_path.relative_to(CELEB_DF_ROOT)
                ).replace("\\", "/"),
                "frame_index": int(frame_index),
                "total_video_frames": int(total_frames),
                "saved_image_path": str(save_path),
                "status": "saved",
            })

            saved_counts[class_name] += 1
            saved_from_video += 1

        capture.release()

        if saved_from_video > 0:
            successful_video_counts[class_name] += 1
        else:
            failed_video_counts[class_name] += 1
            log_rows.append({
                "official_label": int(label),
                "class_name": class_name,
                "source_video": str(video_path),
                "relative_video_path": str(
                    video_path.relative_to(CELEB_DF_ROOT)
                ).replace("\\", "/"),
                "frame_index": None,
                "total_video_frames": int(total_frames),
                "saved_image_path": None,
                "status": "no_face_saved",
            })

    log_df = pd.DataFrame(log_rows)
    log_df.to_csv(
        OUTPUT_ROOT / "extraction_log.csv",
        index=False,
        encoding="utf-8-sig",
    )

    summary = pd.DataFrame([
        {
            "class_name": "real",
            "selected_video_count": sum(
                1 for label, _ in records if label == 1
            ),
            "successful_video_count": (
                successful_video_counts["real"]
            ),
            "failed_video_count": failed_video_counts["real"],
            "saved_image_count": saved_counts["real"],
            "expected_max_images": (
                sum(1 for label, _ in records if label == 1)
                * MAX_FRAMES_PER_VIDEO
            ),
        },
        {
            "class_name": "fake",
            "selected_video_count": sum(
                1 for label, _ in records if label == 0
            ),
            "successful_video_count": (
                successful_video_counts["fake"]
            ),
            "failed_video_count": failed_video_counts["fake"],
            "saved_image_count": saved_counts["fake"],
            "expected_max_images": (
                sum(1 for label, _ in records if label == 0)
                * MAX_FRAMES_PER_VIDEO
            ),
        },
    ])

    summary.to_csv(
        OUTPUT_ROOT / "summary.csv",
        index=False,
        encoding="utf-8-sig",
    )

    return summary


SUMMARY = process_balanced_test_set(
    BALANCED_TEST_RECORDS
)

SUMMARY


處理平衡 Celeb-DF 測試影片:   0%|          | 0/356 [00:00<?, ?it/s]

,class_name,selected_video_count,successful_video_count,failed_video_count,saved_image_count,expected_max_images
0,real,178,178,0,534,534
1,fake,178,178,0,534,534


## 6. 輸出檢查

原本此處為 Real 與 Fake 各 6 張的裁切結果預覽。因 Celeb-DF v2 的授權條款限制影像再散布，已於公開版本移除。程式碼保留，可在本機重現。

In [8]:
import matplotlib.pyplot as plt

def show_samples(folder: Path, title: str, count: int = 6):
    files = sorted(folder.glob("*.png"))[:count]

    if not files:
        print("沒有圖片：", folder)
        return

    fig, axes = plt.subplots(2, 3, figsize=(10, 7))

    for ax, path in zip(axes.flat, files):
        ax.imshow(Image.open(path).convert("RGB"))
        ax.set_title(path.name[:25])
        ax.axis("off")

    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


show_samples(
    OUTPUT_ROOT / "test" / "real",
    "Celeb-DF 官方測試集：Real",
)

show_samples(
    OUTPUT_ROOT / "test" / "fake",
    "Celeb-DF 官方測試集：Fake",
)


## 輸出結果

| 類別 | 影片數 | 成功處理 | 影像數 |
|---|---|---|---|
| Real | 178 | 178 | 534 |
| Fake | 178 | 178 | 534 |

356 部影片全數成功，無讀取失敗或偵測不到人臉的案例。

輸出檔案：

- `test/real/`、`test/fake/`：測試影像
- `selected_fake_videos_seed42.txt`：本次抽中的 178 部 Fake 影片清單
- `balanced_video_list.csv`：本次使用的 356 部影片
- `extraction_log.csv`：每張影像的來源影片與影格位置
- `summary.csv`：影片與影像數量統計

04 與 08 直接讀取 `test/` 目錄。